# Evaluation v3 — Baseline Comparison (revised metrics)

## What changed from v2, and why

Three problems were found in the v2 evaluation code. All three inflated
the score of the B0 traditional baseline.

**1. The style metric used a single reference image.**
v2 compared each output against one NPC portrait. A distance to one image
measures similarity to that character's specific hair and clothing
colours, not to the style as a whole. v3 averages the histogram over the
full 30-image reference set, so individual character differences cancel
out and what remains is what the set has in common.

**2. MAE was computed against the wrong reference.**
v2 compared outputs against `quantized.png`. But B0 is *derived* from
`quantized.png`, so its MAE was measuring pixelation error against its
own input. v3 compares against the input photograph, which is what the
metric claims to measure.

**3. B0 applied a redundant palette remap.**
`pixelate()` uses nearest-neighbour interpolation and therefore creates
no new colours. Remapping the result to the palette it already uses is a
no-op. Removed for clarity.

## Four methods compared

- B0: traditional pixelation only, no learned component
- B1: LoRA only, no ControlNet, no img2img
- B2: LoRA + ControlNet, no colour conditioning
- Ours: LoRA + ControlNet + img2img, complete pipeline

## Run order

Cell 0 → **Restart Runtime** → Cell 1 → Cell 1b → Cell 2 → Cell 3 → Cell 4 → Cell 5 → Cell 6

In [ ]:
# Cell 0: Install
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('[WARN]', r.stderr[-400:])

print('[1/4] huggingface_hub==0.21.4')
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--force-reinstall',
     'huggingface_hub==0.21.4'])

print('[2/4] diffusers / transformers / peft (--no-deps)')
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps',
     'diffusers==0.27.2', 'transformers==4.39.3', 'accelerate==0.29.3',
     'safetensors==0.4.3', 'tokenizers==0.15.2', 'peft==0.10.0',
     'einops', 'omegaconf'])

print('[3/4] metrics + image libraries (no torchvision)')
run([sys.executable, '-m', 'pip', 'install', '--quiet',
     'scikit-image', 'scikit-learn', 'opencv-python-headless', 'Pillow'])

print('[4/4] numpy==1.26.4 (pin last)')
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--force-reinstall',
     'numpy==1.26.4'])

import numpy
print(f'numpy: {numpy.__version__}  (must be 1.26.x)')
print('Done. --> Runtime > Restart Runtime, then run Cell 1 onwards.')

[1/4] huggingface_hub==0.21.4
[2/4] diffusers / transformers / peft (--no-deps)
[3/4] metrics + image libraries (no torchvision)
[4/4] numpy==1.26.4 (pin last)
numpy: 2.0.2  (must be 1.26.x)
Done. --> Runtime > Restart Runtime, then run Cell 1 onwards.


In [ ]:
# Cell 1: Upload the four working files
#
# Upload together:
#   canny_final.png     from Phase 1A  - ControlNet structure input
#   quantized.png       from Phase 1A  - colour reference for img2img
#   portrait_512.png    from Phase 1A  - the white-background input photo
#   stardew_pixel_lora_final.safetensors
#
# Note: the NPC reference set is uploaded separately in Cell 1b.

from google.colab import files
from PIL import Image
import io, os

print('Upload canny_final.png, quantized.png, portrait_512.png, and the LoRA file...')
uploaded = files.upload()

CANNY_PATH = QUANT_PATH = PHOTO_PATH = LORA_PATH = None

for fname, data in uploaded.items():
    path = f'/content/{fname}'
    with open(path, 'wb') as f:
        f.write(data)
    fl = fname.lower()
    if   'canny'     in fl:            CANNY_PATH = path
    elif 'quantized' in fl:            QUANT_PATH = path
    elif 'portrait'  in fl:            PHOTO_PATH = path
    elif fl.endswith('.safetensors'):  LORA_PATH  = path

print(f'  Canny edge map : {CANNY_PATH}')
print(f'  Quantised      : {QUANT_PATH}')
print(f'  Input photo    : {PHOTO_PATH}')
print(f'  LoRA weights   : {LORA_PATH}')

assert all([CANNY_PATH, QUANT_PATH, PHOTO_PATH, LORA_PATH]), \
    'Missing a file - check filenames contain canny / quantized / portrait'

Upload canny_final.png, quantized.png, portrait_512.png, and the LoRA file...


Saving canny_final (2).png to canny_final (2).png
Saving portrait_512.png to portrait_512.png
Saving quantized (2).png to quantized (2).png
Saving stardew_pixel_lora_final (5).safetensors to stardew_pixel_lora_final (5).safetensors
  Canny edge map : /content/canny_final (2).png
  Quantised      : /content/quantized (2).png
  Input photo    : /content/portrait_512.png
  LoRA weights   : /content/stardew_pixel_lora_final (5).safetensors


In [ ]:
# Cell 1b: Upload the NPC reference set
#
# WHY THIS CELL EXISTS
# v2 compared every output against a single NPC portrait. The histogram of
# one image is dominated by that character's own hair, skin and clothing
# colours, so the metric was measuring "how similar is this to Dobson"
# rather than "how similar is this to the Stardew Valley style".
#
# Averaging over the whole set cancels individual character differences.
# What survives the average is what the portraits share: the saturation
# range, the hue distribution, the proportion of white background.
#
# Upload the same ZIP you used for LoRA training. The .txt caption files
# are ignored; only images are collected.

from google.colab import files
import zipfile, os, glob, shutil

REF_DIR = '/content/npc_reference'
TMP_DIR = '/content/_ref_tmp'
for d in (REF_DIR, TMP_DIR):
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)

print('Upload the ZIP of your 30 training portraits...')
uploaded = files.upload()

zips = [f for f in uploaded if f.lower().endswith('.zip')]
assert zips, 'No .zip found in the upload'

with zipfile.ZipFile(f'/content/{zips[0]}', 'r') as z:
    z.extractall(TMP_DIR)

# Flatten: collect every image regardless of folder depth, ignore captions
n = 0
for root, _, fnames in os.walk(TMP_DIR):
    for fn in fnames:
        if fn.lower().endswith(('.png', '.jpg', '.jpeg')):
            shutil.copy(os.path.join(root, fn),
                        os.path.join(REF_DIR, f'{n:03d}_{fn}'))
            n += 1

REF_PATHS = sorted(glob.glob(f'{REF_DIR}/*'))
print(f'\nReference set: {len(REF_PATHS)} images collected')
assert len(REF_PATHS) >= 10, \
    f'Only {len(REF_PATHS)} images found - need at least 10 for a meaningful average'

Upload the ZIP of your 30 training portraits...


Saving train.zip to train.zip

Reference set: 30 images collected


In [ ]:
# Cell 2: Load shared resources
import subprocess, sys
# Re-pin numpy before importing diffusers - Colab reloads system numpy on restart
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--force-reinstall', 'numpy==1.26.4'], capture_output=True)

import torch
import numpy as np
from PIL import Image
from diffusers import (
    ControlNetModel,
    StableDiffusionPipeline,
    StableDiffusionControlNetPipeline,
    StableDiffusionControlNetImg2ImgPipeline,
    UniPCMultistepScheduler,
)
from safetensors.torch import load_file

BASE_MODEL = 'runwayml/stable-diffusion-v1-5'
CTRL_ID    = 'lllyasviel/sd-controlnet-canny'
LORA_SCALE = 0.8
CN_SCALE   = 0.8
STRENGTH   = 0.6
SEED       = 42
PIXEL_SIZE = 8

PROMPT     = ('stardew_pixel_art, 1girl, solo, portrait, '
              'white background, upper body, facing viewer, pixel art')
NEG_PROMPT = ('red background, gray background, gradient, '
              'blurry, realistic, 3d, multiple characters')


def merge_lora(unet, path, scale):
    """Merge LoRA residuals into base weights: W_new = W + scale * (B @ A).

    Handles both 2D linear projections and 4D unit-kernel convolutions by
    flattening both learned matrices to 2D, multiplying, then reshaping.
    """
    w  = load_file(path)
    sd = unet.state_dict()
    n  = 0
    for a_key in [k for k in w if k.endswith('lora_A.default.weight')]:
        b_key    = a_key.replace('lora_A', 'lora_B')
        base_key = a_key.replace('base_model.model.', '') \
                        .replace('.lora_A.default.weight', '.weight')
        if b_key not in w or base_key not in sd:
            continue
        A = w[a_key].float().cpu()
        B = w[b_key].float().cpu()
        o = sd[base_key].float().cpu()
        sd[base_key] = o + (B.view(B.shape[0], -1) @
                            A.view(A.shape[0], -1)).view(o.shape) * scale
        n += 1
    unet.load_state_dict(sd, strict=True)
    print(f'  LoRA merged: {n} weights')
    return unet.half()


def pixelate(img, block=8):
    """Nearest-neighbour downsample then upsample, producing uniform cells.

    Nearest-neighbour creates no new colours, it only replicates existing
    ones. This is why a subsequent palette remap would be a no-op.
    """
    w, h = img.size
    return img.resize((w // block, h // block),
                      Image.NEAREST).resize((w, h), Image.NEAREST)


print('Loading ControlNet (~700 MB on first run)...')
controlnet = ControlNetModel.from_pretrained(CTRL_ID, torch_dtype=torch.float16)
print('ControlNet loaded.')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


Loading ControlNet (~700 MB on first run)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ControlNet loaded.


In [ ]:
# Cell 3: Generate all four baseline outputs
import torch
from PIL import Image

canny_img = Image.open(CANNY_PATH).convert('RGB').resize((512, 512))
quant_img = Image.open(QUANT_PATH).convert('RGB').resize((512, 512))
results   = {}

# B0: traditional pixelation, no learned component
# Quantisation already happened in Phase 1A, so this stage is only the
# downsample/upsample step. The v2 palette remap was removed: nearest-
# neighbour interpolation produces no new colours, so remapping to the
# palette the image already uses changes nothing.
print('B0: traditional pixelation...')
b0 = pixelate(quant_img, PIXEL_SIZE)
b0.save('/content/B0_traditional.png')
results['B0_Traditional'] = b0

# B1: LoRA only - text-to-image, no structural or colour input
print('B1: LoRA only...')
pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, safety_checker=None).to('cuda')
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.unet      = merge_lora(pipe.unet, LORA_PATH, LORA_SCALE)
with torch.autocast('cuda'):
    b1 = pipe(PROMPT, negative_prompt=NEG_PROMPT,
              num_inference_steps=30, guidance_scale=7.5,
              generator=torch.Generator('cuda').manual_seed(SEED),
              width=512, height=512).images[0]
b1.save('/content/B1_LoRA_only.png')
results['B1_LoRA_only'] = b1
del pipe; torch.cuda.empty_cache()

# B2: LoRA + ControlNet - structure but no colour reference
print('B2: LoRA + ControlNet...')
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    BASE_MODEL, controlnet=controlnet,
    torch_dtype=torch.float16, safety_checker=None).to('cuda')
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.unet      = merge_lora(pipe.unet, LORA_PATH, LORA_SCALE)
with torch.autocast('cuda'):
    b2 = pipe(PROMPT, negative_prompt=NEG_PROMPT,
              image=canny_img,
              controlnet_conditioning_scale=CN_SCALE,
              num_inference_steps=30, guidance_scale=7.5,
              generator=torch.Generator('cuda').manual_seed(SEED),
              width=512, height=512).images[0]
b2.save('/content/B2_LoRA_ControlNet.png')
results['B2_LoRA+ControlNet'] = b2
del pipe; torch.cuda.empty_cache()

# Ours: LoRA + ControlNet + img2img colour conditioning
print('Ours: complete pipeline...')
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    BASE_MODEL, controlnet=controlnet,
    torch_dtype=torch.float16, safety_checker=None).to('cuda')
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.unet      = merge_lora(pipe.unet, LORA_PATH, LORA_SCALE)
with torch.autocast('cuda'):
    ours = pipe(PROMPT, negative_prompt=NEG_PROMPT,
                image=quant_img, control_image=canny_img,
                controlnet_conditioning_scale=CN_SCALE,
                strength=STRENGTH,
                num_inference_steps=30, guidance_scale=7.5,
                generator=torch.Generator('cuda').manual_seed(SEED)).images[0]
ours.save('/content/Ours_complete.png')
results['Ours_Complete'] = ours
del pipe; torch.cuda.empty_cache()

print('\nAll four methods generated.')

B0: traditional pixelation...
B1: LoRA only...


You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


  LoRA merged: 160 weights


B2: LoRA + ControlNet...


You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet.StableDiffusionControlNetPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


  LoRA merged: 160 weights


Ours: complete pipeline...


You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet_img2img.StableDiffusionControlNetImg2ImgPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


  LoRA merged: 160 weights



All four methods generated.


In [ ]:
# Cell 4: Compute metrics  [REVISED]
#
# What each metric is computed against
#
#   SSIM       output vs input photograph          higher is better
#   StyleDist  output vs 30-image reference set    lower  is better
#   MAE        output vs input photograph          lower  is better
#
# Changes from v2
#
#   StyleDist replaces Hist-dist. Same underlying computation - an L1
#   distance between normalised HSV histograms - but the reference is
#   now the mean histogram of the whole training set rather than one
#   image. Renamed because "histogram distance" describes the mechanism
#   while "style distance" describes what it is meant to measure.
#
#   MAE now uses the input photograph as reference. v2 used
#   quantized.png, from which B0 is directly derived, so B0's MAE was
#   measuring its own pixelation error rather than colour fidelity.

# Recover REF_PATHS from disk in case the variable was lost
import glob
REF_PATHS = sorted(glob.glob('/content/npc_reference/*'))
print(f'Reference set: {len(REF_PATHS)} images')
assert len(REF_PATHS) >= 10, 'Reference dir is empty - re-run Cell 1b'

import numpy as np
from PIL import Image
from skimage.metrics import structural_similarity as ssim
import cv2


def hsv_hist(img, bins=32):
    """Normalised hue+saturation histogram of one RGB PIL image.

    Hue and saturation are used rather than RGB because they separate
    colour identity from brightness, which makes the measure less
    sensitive to overall lighting differences between photograph and
    hand-drawn reference.
    """
    bgr  = cv2.cvtColor(np.array(img.resize((256, 256))), cv2.COLOR_RGB2BGR)
    hsv  = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    h    = cv2.calcHist([hsv], [0], None, [bins], [0, 180])
    s    = cv2.calcHist([hsv], [1], None, [bins], [0, 256])
    hist = np.concatenate([h, s]).flatten()
    return hist / (hist.sum() + 1e-7)


# Build the reference distribution once, from the whole set
print(f'Building reference distribution from {len(REF_PATHS)} images...')
ref_hists = [hsv_hist(Image.open(p).convert('RGB')) for p in REF_PATHS]
ref_mean  = np.mean(ref_hists, axis=0)

# Spread within the reference set - useful context for interpreting the
# distances below. If the outputs are further from the mean than the
# reference images are from each other, they fall outside the style.
ref_spread = float(np.mean([np.abs(h - ref_mean).sum() for h in ref_hists]))
print(f'Mean within-set distance: {ref_spread:.4f}')
print('  (reference images are on average this far from their own mean)')
print()


def style_distance(img):
    return float(np.abs(hsv_hist(img.convert('RGB')) - ref_mean).sum())


photo_np = np.array(Image.open(PHOTO_PATH).convert('RGB').resize((512, 512)))

rows = []
print(f'{"Method":<22} {"SSIM":>8} {"StyleDist":>11} {"MAE":>8}')
print('-' * 52)

for name, img in results.items():
    gen_np = np.array(img.convert('RGB').resize((512, 512)))

    s = ssim(photo_np, gen_np, channel_axis=2, data_range=255)
    d = style_distance(img)
    m = np.abs(gen_np.astype(float) - photo_np.astype(float)).mean()

    rows.append((name, s, d, m))
    print(f'{name:<22} {s:>8.4f} {d:>11.4f} {m:>8.2f}')

print('-' * 52)
print('Reference for each metric:')
print('  SSIM       input photograph      higher = structure preserved')
print('  StyleDist  30-image NPC set      lower  = closer to target style')
print('  MAE        input photograph      lower  = colours match the photo')
print()
print('Interpretation note: SSIM and MAE both measure fidelity to the')
print('input. A method that leaves the input unchanged scores best on')
print('both by construction, regardless of whether any stylistic')
print('transformation has occurred. StyleDist is the only measure here')
print('that compares against the target style rather than the source.')

Reference set: 30 images
Building reference distribution from 30 images...
Mean within-set distance: 0.5352
  (reference images are on average this far from their own mean)

Method                     SSIM   StyleDist      MAE
----------------------------------------------------
B0_Traditional           0.7424      0.9120     8.39
B1_LoRA_only             0.2955      1.1428   107.09
B2_LoRA+ControlNet       0.2371      1.3470   121.34
Ours_Complete            0.6954      0.9091    17.78
----------------------------------------------------
Reference for each metric:
  SSIM       input photograph      higher = structure preserved
  StyleDist  30-image NPC set      lower  = closer to target style
  MAE        input photograph      lower  = colours match the photo

Interpretation note: SSIM and MAE both measure fidelity to the
input. A method that leaves the input unchanged scores best on
both by construction, regardless of whether any stylistic
transformation has occurred. StyleDist is th

In [ ]:
# Cell 4b: Perceptual distance in CLIP feature space
#
# WHY THIS CELL EXISTS
# The HSV histogram in Cell 4 could not separate B0 from the complete
# pipeline (0.9120 vs 0.9091). This is expected: a histogram compares
# how much of each colour is present, not how those colours are
# arranged. A pixelated photograph and a stylised portrait can have
# almost identical colour proportions while differing completely in
# spatial organisation - flat regions versus retained gradients,
# simplified features versus photographic detail.
#
# CLIP image embeddings are sensitive to both texture and structure,
# so they can register differences the histogram cannot. The same
# approach is used as a subject-fidelity measure in the DreamBooth
# literature.
#
# Two axes are computed:
#   StyleDist_CLIP     mean distance to the 30 reference portraits
#   IdentityDist_CLIP  distance to the input photograph
#
# Unlike FID or KID, this is a pairwise comparison rather than a
# distribution estimate, so it remains valid at N=30.

import torch
import numpy as np
from PIL import Image
from transformers import CLIPModel, CLIPImageProcessor

print('Loading CLIP image encoder (~600 MB on first run)...')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').eval()
clip_proc  = CLIPImageProcessor.from_pretrained('openai/clip-vit-base-patch32')


def clip_embed(img):
    """Return the L2-normalised CLIP image embedding of one PIL image."""
    inputs = clip_proc(images=img.convert('RGB'), return_tensors='pt')
    with torch.no_grad():
        feat = clip_model.get_image_features(**inputs)
    return (feat / feat.norm(dim=-1, keepdim=True)).squeeze(0)


# Embed the reference set once
print(f'Embedding {len(REF_PATHS)} reference portraits...')
ref_embeds = torch.stack([clip_embed(Image.open(p)) for p in REF_PATHS])

# Embed the input photograph
photo_embed = clip_embed(Image.open(PHOTO_PATH))

# Spread within the reference set, for scale reference.
# Each portrait's mean distance to the other 29.
n = len(ref_embeds)
sim_matrix = ref_embeds @ ref_embeds.T                # cosine similarities
off_diag   = sim_matrix[~torch.eye(n, dtype=bool)]    # exclude self-similarity
ref_spread_clip = float(1 - off_diag.mean())

print(f'\nMean within-set CLIP distance: {ref_spread_clip:.4f}')
print('  (reference portraits are on average this far from each other)')
print()

clip_rows = []
print(f'{"Method":<22} {"StyleDist_CLIP":>15} {"IdentityDist_CLIP":>19}')
print('-' * 58)

for name, img in results.items():
    e = clip_embed(img)
    style_d    = float(1 - (ref_embeds @ e).mean())   # vs reference set
    identity_d = float(1 - torch.dot(photo_embed, e)) # vs input photograph
    clip_rows.append((name, style_d, identity_d))
    print(f'{name:<22} {style_d:>15.4f} {identity_d:>19.4f}')

print('-' * 58)
print('StyleDist_CLIP     lower = closer to the target style')
print('IdentityDist_CLIP  lower = closer to the input photograph')
print()
print(f'Scale reference: portraits within the training set differ from')
print(f'each other by {ref_spread_clip:.4f} on the same measure. An output')
print('scoring near or below that value sits inside the style; one')
print('scoring well above it sits outside.')

Loading CLIP image encoder (~600 MB on first run)...


Embedding 30 reference portraits...

Mean within-set CLIP distance: 0.1797
  (reference portraits are on average this far from each other)

Method                  StyleDist_CLIP   IdentityDist_CLIP
----------------------------------------------------------
B0_Traditional                  0.4068              0.1647
B1_LoRA_only                    0.1645              0.5503
B2_LoRA+ControlNet              0.2285              0.4625
Ours_Complete                   0.2849              0.3281
----------------------------------------------------------
StyleDist_CLIP     lower = closer to the target style
IdentityDist_CLIP  lower = closer to the input photograph

Scale reference: portraits within the training set differ from
each other by 0.1797 on the same measure. An output
scoring near or below that value sits inside the style; one
scoring well above it sits outside.


In [ ]:
# Cell 4c: Style-identity trade-off plot
#
# The two CLIP measures from Cell 4b are plotted against each other.
# Each configuration is one point. The dashed lines mark the mean
# within-set distance of the reference portraits, which gives the
# axes a scale: a point below a dashed line is closer to that target
# than the reference portraits are to each other.

# Recover the Cell 4b results in case the kernel was restarted
try:
    clip_rows
    print('Using clip_rows from Cell 4b')
except NameError:
    clip_rows = [
        ('B0_Traditional',     0.4068, 0.1647),
        ('B1_LoRA_only',       0.1645, 0.5503),
        ('B2_LoRA+ControlNet', 0.2285, 0.4625),
        ('Ours_Complete',      0.2849, 0.3281),
    ]
    ref_spread_clip = 0.1797
    print('Restored clip_rows from recorded values')

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6.5))

style_vals    = [r[1] for r in clip_rows]
identity_vals = [r[2] for r in clip_rows]
labels        = [r[0] for r in clip_rows]

# Order the points along the frontier so the connecting line is monotonic
order = sorted(range(len(labels)), key=lambda i: style_vals[i])
ax.plot([style_vals[i] for i in order],
        [identity_vals[i] for i in order],
        '--', color='#999999', linewidth=1.2, zorder=1)

colours = {
    'B0_Traditional':     '#D55E00',
    'B1_LoRA_only':       '#0072B2',
    'B2_LoRA+ControlNet': '#009E73',
    'Ours_Complete':      '#CC0000',
}

for name, sx, iy in clip_rows:
    is_ours = name == 'Ours_Complete'
    ax.scatter(sx, iy,
               s=260 if is_ours else 170,
               marker='*' if is_ours else 'o',
               color=colours.get(name, '#333333'),
               edgecolors='black', linewidths=1.1, zorder=3)
    ax.annotate(name.replace('_', ' '),
                (sx, iy), textcoords='offset points',
                xytext=(11, 9), fontsize=10,
                fontweight='bold' if is_ours else 'normal')

# Scale references
ax.axvline(ref_spread_clip, ls=':', color='#666666', lw=1.1)
ax.axhline(ref_spread_clip, ls=':', color='#666666', lw=1.1)
ax.text(ref_spread_clip, ax.get_ylim()[1] * 0.98,
        f'  within-set spread = {ref_spread_clip:.3f}',
        rotation=90, va='top', fontsize=8.5, color='#666666')

ax.set_xlabel('StyleDist (CLIP)  -  distance to reference portraits'
              '\nlower = closer to Stardew Valley style', fontsize=11)
ax.set_ylabel('IdentityDist (CLIP)  -  distance to input photograph'
              '\nlower = closer to the person in the photo', fontsize=11)
ax.set_title('Style-identity trade-off across four configurations',
             fontsize=13, fontweight='bold', pad=14)
ax.grid(alpha=0.25, linestyle='-', linewidth=0.5)

ax.text(0.02, 0.02, 'ideal region\n(both low)', transform=ax.transAxes,
        fontsize=9, style='italic', color='#444444',
        ha='left', va='bottom')

plt.tight_layout()
plt.savefig('/content/tradeoff_plot.png', dpi=200, bbox_inches='tight')
plt.show()

print('Saved: /content/tradeoff_plot.png')
print()
print('Reading the plot:')
print('  Moving left  = better style match')
print('  Moving down  = better identity preservation')
print('  The four points form a monotonic frontier: every gain on one')
print('  axis costs something on the other. No configuration dominates.')

Restored clip_rows from recorded values


Saved: /content/tradeoff_plot.png

Reading the plot:
  Moving left  = better style match
  Moving down  = better identity preservation
  The four points form a monotonic frontier: every gain on one
  axis costs something on the other. No configuration dominates.


In [ ]:
# Cell 5: Comparison figure
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

cols = [
    ('Input photograph',                       PHOTO_PATH),
    ('B0: Traditional\n(no learned component)', '/content/B0_traditional.png'),
    ('B1: LoRA only',                          '/content/B1_LoRA_only.png'),
    ('B2: LoRA + ControlNet',                  '/content/B2_LoRA_ControlNet.png'),
    ('Ours: Complete pipeline',                '/content/Ours_complete.png'),
]

fig, axes = plt.subplots(1, 5, figsize=(25, 6))
for ax, (label, path) in zip(axes, cols):
    ax.imshow(mpimg.imread(path))
    ax.set_title(label, fontsize=11)
    ax.axis('off')

score_line = '     '.join(
    f'{r[0]}:  SSIM {r[1]:.3f}  |  Style {r[2]:.3f}  |  MAE {r[3]:.1f}'
    for r in rows)
plt.suptitle('Baseline comparison\n' + score_line, fontsize=8, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 6: Metric table figure + download everything
import matplotlib.pyplot as plt
import os

# Merge the histogram metrics from Cell 4 with the CLIP metrics from Cell 4b
merged = []
for (name, s, d, m), (_, cs, ci) in zip(rows, clip_rows):
    merged.append([name, f'{s:.4f}', f'{d:.4f}', f'{m:.2f}',
                   f'{cs:.4f}', f'{ci:.4f}'])

fig, ax = plt.subplots(figsize=(13, 2.6))
ax.axis('off')

table = ax.table(
    cellText=merged,
    colLabels=['Configuration', 'SSIM\n(higher)', 'StyleDist HSV\n(lower)',
               'MAE\n(lower)', 'StyleDist CLIP\n(lower)', 'IdentityDist CLIP\n(lower)'],
    cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2.0)
for i in range(6):
    table[(0, i)].set_facecolor('#E8E8E8')
    table[(0, i)].set_text_props(weight='bold')

plt.title('Table 6.1  Quantitative comparison across four configurations',
          fontsize=11, pad=16)
plt.tight_layout()
plt.savefig('/content/metric_table.png', dpi=200, bbox_inches='tight')
plt.show()

from google.colab import files
for p in ['/content/baseline_comparison.png',
          '/content/metric_table.png',
          '/content/B0_traditional.png',
          '/content/B1_LoRA_only.png',
          '/content/B2_LoRA_ControlNet.png',
          '/content/Ours_complete.png']:
    if os.path.exists(p):
        files.download(p)
print('Download complete.')

Download complete.
